# Video Model Traning

## Module Imports

In [ ]:
!pip install -q transformers datasets torchaudio accelerate evaluate soundfile

In [ ]:
import os

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [ ]:
import torch
import numpy as np
from datasets import load_dataset, Audio
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
MODEL_NAME = "facebook/wav2vec2-base"   # ~95M params, fast, good accuracy for spoof detection
DATASET_NAME = "abdulahh35/ANC-Spoof"   # hosts ASVspoof2019 LA (+ FoR, InTheWild) as separate configs
DATASET_CONFIG = "ASVspoof2019"         # we only want this config — the other two are cross-eval-only sets
SAMPLE_RATE = 16000                     # wav2vec2 expects 16kHz audio
MAX_DURATION_SECONDS = 4                # clip/pad every sample to 4s -> keeps training fast on free GPU
OUTPUT_DIR = "./audio_deepfake_model"

In [ ]:
# streaming=True means samples are downloaded one-by-one as needed during training,
# instead of the whole ~100GB+ dataset being downloaded to disk first. This is essential
# on Colab/Kaggle free tier where disk space is limited (usually ~20-100GB total).
# Trade-off: no random-access indexing, no len(), and shuffling only works on a buffer
# (see .shuffle(buffer_size=...) below) instead of the whole dataset at once.
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, streaming=True)
 
# We only want the "Original" codec rows (uncompressed) for training — the dataset also
# includes the same utterances re-encoded through 7 other neural codecs for robustness
# testing, which would inflate/skew a first training run.
# NOTE: with streaming, we can no longer call features["label"].int2str() on the fly the
# same way, so we hardcode the known label mapping instead (confirmed from the dataset card).
dataset = dataset.filter(lambda x: x["codec"] == "Original")
 
# Resample audio column to 16kHz (required by wav2vec2) — this works the same in streaming mode
dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
 
LABEL2ID = {"real": 0, "fake": 1}
ID2LABEL = {0: "real", 1: "fake"}
 
# IterableDataset (streaming) features still carry the ClassLabel definition, so we can
# grab int2str from the "train" split's schema without materializing any data.
_label_feature = dataset["train"].features["label"]
 
def normalize_label(example):
    label_str = _label_feature.int2str(example["label"]).lower()
    is_fake = "spoof" in label_str
    example["labels"] = LABEL2ID["fake"] if is_fake else LABEL2ID["real"]
    return example
 
dataset = dataset.map(normalize_label)
 
# Streaming datasets can only shuffle within a local buffer (pulled progressively), not
# the entire dataset at once — still gives good randomness for training.
dataset["train"] = dataset["train"].shuffle(buffer_size=2000, seed=42)

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
 
def preprocess(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        max_length=int(SAMPLE_RATE * MAX_DURATION_SECONDS),
        truncation=True,
        padding="max_length",
    )
    return inputs
 
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=["audio", "codec", "subset", "split", "speaker", "utt_id"],
    batched=True,
    batch_size=32,
)
# NOTE: with streaming, .map() is lazy — nothing is actually processed until the Trainer
# starts pulling batches during training. This is expected and fine.

In [ ]:
model = AutoModelForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
).to(device)
 
# Freeze the feature encoder (CNN front-end) — speeds up training a lot and
# prevents overfitting on small free-tier GPU sessions. Only the transformer
# layers + classification head get fine-tuned.
model.freeze_feature_encoder()
 
# %% [Cell 7] Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

In [ ]:
# IMPORTANT: streaming (IterableDataset) has no len(), so we can't use num_train_epochs.
# We use max_steps instead — pick a number based roughly on: 
#   steps ≈ (desired_samples_seen) / (batch_size * gradient_accumulation_steps)
# 8000 steps * effective batch 16 ≈ 128,000 samples seen — a reasonable first run.
# Increase MAX_STEPS if you want to train longer / have GPU time to spare.
MAX_STEPS = 8000
EVAL_EVERY_N_STEPS = 500
 
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=EVAL_EVERY_N_STEPS,
    save_strategy="steps",
    save_steps=EVAL_EVERY_N_STEPS,
    learning_rate=2e-5,                # lower LR since we're fine-tuning an already-tuned model
    per_device_train_batch_size=48,    # was 16 — push higher since VRAM headroom is available
    per_device_eval_batch_size=48,
    gradient_accumulation_steps=1,     # no longer needed at this batch size
    max_steps=MAX_STEPS,               # replaces num_train_epochs for streaming datasets
    warmup_steps=50,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                         # mixed precision -> fits comfortably in free GPU VRAM
    dataloader_num_workers=4,          # parallel data loading so GPU doesn't wait on the stream
    dataloader_prefetch_factor=4,      # pre-fetch batches ahead of time
    report_to="none",
)

In [ ]:
# Cap the eval set to a fixed number of samples — with streaming, iterating the FULL
# 199k-row "dev" split on every eval cycle would be extremely slow and defeats the
# purpose of streaming. take(N) pulls just N samples from the stream.
EVAL_SAMPLE_COUNT = 1000
eval_subset = encoded_dataset["dev"].take(EVAL_SAMPLE_COUNT)
 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=eval_subset,
    compute_metrics=compute_metrics,
)
 
trainer.train()

In [ ]:
trainer.save_model(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")


In [ ]:
 # Optional: push to your own Hugging Face Hub repo so you can load it later
# from huggingface_hub import login
# login()  # paste your HF token
# trainer.push_to_hub("your-username/audio-deepfake-detector")